# Create or append Virtual Icechunk Store from SHYFEM forecast NetCDF files

* This notebook appends Taranto SHYFEM forecast data to an Icechunk store in the **rswain** bucket using date-based set logic.
* Source NetCDF files are read from `protocoast-data`; the Icechunk repo is written to `rswain`.
* If no repo exists, a new one will be created with references to all the existing NetCDF files. 

**Append Methodology:**
1. **`set_repo`**: extract all dates currently present in the Icechunk store's `time` coordinate
2. **`set_cloud`**: Scan the S3 bucket for all available NOS files and extract their dates.
3. **`new_dates`**: Calculate the difference (`set_cloud - set_repo`) to determine exactly which days need to be written for creation or appended.

In [1]:
import warnings
import os
import pandas as pd
import fsspec
import xarray as xr
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import icechunk
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

In [3]:
# Load credentials
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/rsignell.env', override=True)

In [4]:
# Configuration
storage_endpoint = os.environ['ENDPOINT_URL']
icechunk_bucket = 'rswain'           # bucket where the Icechunk repo is stored
source_bucket = 'rswain'    # bucket where source NetCDF files live
storage_name = 'ibi-wave-2017-2024-v1'
icechunk_bucket_url = f"s3://{icechunk_bucket}"
source_bucket_url = f"s3://{source_bucket}"

# Setup Filesystem (for scanning source files)
fs = fsspec.filesystem('s3', anon=False, endpoint_url=storage_endpoint,
                       skip_instance_cache=True, use_listings_cache=False)

In [5]:
fs.ls(icechunk_bucket)

['rswain/icechunk', 'rswain/testing']

In [6]:
fs.ls(f'{icechunk_bucket_url}/icechunk')

['rswain/icechunk/ibi-wave-2017-2024-v1',
 'rswain/icechunk/taranto-icechunk-tubitak-v1']

In [7]:
icechunk_url = f'{icechunk_bucket_url}/icechunk/{storage_name}'
icechunk_url

's3://rswain/icechunk/ibi-wave-2017-2024-v1'

In [8]:
fs.du(icechunk_url)/1e9

54.38986256

In [ ]:
# Define Icechunk Storage & Config
# Repo is written to the rswain bucket
storage = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()
# Virtual chunks point back to the source data in protocoast-data
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{source_bucket_url}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True,
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"{source_bucket_url}/": icechunk.s3_credentials(anonymous=False)})

store_obj = S3Store(
    bucket=source_bucket,
    endpoint=storage_endpoint,
    region="not-used",
)

registry = ObjectStoreRegistry({source_bucket_url: store_obj})
parser = HDFParser()

### Step 1: Create Date Sets (`set_repo` vs `set_cloud`)

In [ ]:
# --- 1. Get Dates from Icechunk Repo (set_repo) ---
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, consolidated=False, chunks={})
    
    if 'time' in ds.coords:
        # Extract dates as YYYYMMDD strings
        dates = pd.to_datetime(ds.time.values) + pd.Timedelta(days=1)
        set_repo = set(dates.strftime('%Y%m%d'))
    else:
        set_repo = set()
        
except Exception as e:
    print(f"Repo access failed or empty ({e}). Assuming set_repo is empty.")
    repo = None
    set_repo = set()

print(f"set_repo: {len(set_repo)} dates found.")

# --- 2. Get Dates from Cloud Bucket (set_cloud) ---
print("Scanning S3 for NOS files...")
nos_files = fs.glob(f'{source_bucket}/full_dataset/shyfem/taranto/forecast/*/*nos*.nc')

set_cloud = set()
date_to_files_map = {}

for f in nos_files:
    try:
        date_str = f.split('/')[-2]
        set_cloud.add(date_str)
        
        base_dir = os.path.dirname(f)
        nos_path = f's3://{f}'
        ous_path = f's3://{base_dir}/taranto_ous_{date_str}_nc4.nc'
        
        date_to_files_map[date_str] = {'nos': nos_path, 'ous': ous_path}
        
    except IndexError:
        pass

print(f"set_cloud: {len(set_cloud)} dates found.")

### Step 3: Append to Icechunk

In [ ]:
if ds_final is not None:
    if repo is None:
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        initial_session = repo.writable_session("main")

        print(f"Writing {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(initial_session.store)
    
        msg = f"Initialized with forecast data: {new_dates[0]} to {new_dates[-1]}"
        initial_session.commit(msg)
        print(f"Commit successful: '{msg}'")
    else:
        append_session = repo.writable_session("main")

        print(f"Appending {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(append_session.store, append_dim="time")
    
        msg = f"Appended forecast data: {new_dates[0]} to {new_dates[-1]}"
        append_session.commit(msg)
        print(f"Commit successful: '{msg}'")

    history = repo.ancestry(branch="main")
    latest = next(history)
    print(f"Latest Commit [{latest.written_at}]: {latest.message}")
    
else:
    print("Nothing to append.")

In [ ]:
nos_files

In [ ]:
xr.open_dataset(fs.open(nos_files[0]))

In [ ]:
ds_final

In [9]:
storage2 = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)
repo2 = icechunk.Repository.open(storage2)
session2 = repo2.readonly_session("main")
ds2 = xr.open_zarr(session2.store, consolidated=False, chunks={})
ds2

<xarray.Dataset> Size: 123GB
Dimensions:    (time: 864, latitude: 1081, longitude: 865)
Coordinates:
  * latitude   (latitude) float32 4kB 26.0 26.03 26.06 ... 55.95 55.97 56.0
  * longitude  (longitude) float32 3kB -19.0 -18.97 -18.94 ... 4.945 4.973 5.001
  * time       (time) datetime64[ns] 7kB 2022-11-26 ... 2022-12-31T23:00:00
Data variables: (12/19)
    VCMX       (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VHM0_SW1   (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VHM0_WW    (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VMDR_SW1   (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VHM0_SW2   (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VTM01_SW2  (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    ...         ...
    VTPK       (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VTM10      (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VMXL       (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VHM0       (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VTM01_SW1  (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
    VTM02      (time, latitude, longitude) float64 6GB dask.array<chunksize=(100, 1081, 865), meta=np.ndarray>
Attributes: (12/14)
    Conventions:               CF-1.8
    comment:                   
    contact:                   https://marine.copernicus.eu/contact
    domain_name:               IBI36
    field_date:                20241102
    field_type:                instantaneous
    ...                        ...
    institution:               NOWSystems-MeteoFrance
    licence:                   https://marine.copernicus.eu/user-corner/servi...
    references:                http://marine.copernicus.eu/
    source:                    MFWAM-CY47R1
    title:                     Wave hourly instantaneous fields for the Iberi...
    copernicusmarine_version:  2.3.0